# 05 — qwen7b SFT adapters, TQ1_0

Select a **Colab GPU** kernel in VS Code. Read `04_quantized_setup.ipynb` for the laptop receiver, tunnel, and audible watchdog. Use only the pinned frozen base/adapters. Inference sends the same complete, unmodified teacher-row prompt with the frozen chat template, greedy decoding, seed 42, repetition penalty 1 and 1,024-token maximum. Every row is an independent llama-server completion with prompt caching disabled; prompt identity and tokenization are checked. An empty or malformed model response is still a completed response. Context-unsupported rows retain an explicit status. This condition does not use CAM-LDS importance-matrix calibration.

**Inference process revision (2 November 2026).** Earlier saved 04 responses used one llama-server slot and synchronously acknowledged each row through the laptop receiver. The revised process uses model-specific concurrent slots for independent one-row requests, atomic Colab records, background receiver uploads in batches of up to five, backpressure at 25 outstanding rows, and laptop CSV snapshots every 25 mutations. GGUF construction, rendered prompts, frozen HF token IDs, per-row context allowance, and generation settings are unchanged. The old and new processes can coexist within the same resumable stream; the per-row parallel_slots field distinguishes new records. Before the revision, this 05 notebook had 0/285 response rows saved for each of its four adapter streams, according to results/quantized record files. This refers to inference results, not preparation of adapter GGUF artifacts.

**L4 runtime revision (4 November 2026).** This revision applies only to the unfinished qwen7b 05 notebook; Hermes was already complete and Qwen1B was already running, so neither is changed. At adoption, all four qwen7b adapter streams held 0/285 response rows. The pinned llama.cpp toolchain is compiled in Release mode for the allocated GPU architecture (L4 compute capability 8.9 / `sm_89`) and a matching saved toolchain is restored when available. Completed GGUF artifacts are restored from the laptop instead of being rebuilt. Each adapter uses one persistent server with 2 independent slots, batch 2,048, microbatch 512, backend-default Flash Attention and backend-default CPU threads. Missing prompts are ordered by frozen prompt-token length using fixed buckets without changing prompt text, token IDs, contexts or decoding. The five-row pilot retains diagnostic verbosity 2; the full run uses verbosity 1 while retaining per-row records, errors and final telemetry. Run only one inference notebook per L4. Atomic local records, five-row background receiver batches and 25-row backpressure remain enabled. Sampling, natural EOS and the 1,024-token ceiling are unchanged.

**Single-adapter parallel-run copy.** This notebook runs only the `joint` adapter. It uses the identical artifact, stream identity, prompts, inference settings and resumable laptop checkpoints as the preserved combined 05 notebook. Run different adapter copies on separate Colab runtimes; never run this copy and the combined notebook on the same adapter simultaneously.

## 1. Restore code and data from laptop

Start the extended receiver, tunnel and watchdog using notebook 04 setup. Paste the HTTPS URL and receiver token. The receiver sends frozen code, manifests, prompts and tokenizer files; it does not send API/HF secrets in the bundle.

**Recovery / safe rerun:** New runtime or tunnel: rerun 1, then 2, 3, and the interrupted stage. The same URL/token may be reused across different models.

**Cost:** No paid API requests.



In [2]:
from pathlib import Path
from getpass import getpass
import json, io, os, sys, urllib.request, zipfile
ROOT = Path('/content/CAM_LDS_STREAMLINED')
ROOT.mkdir(parents=True, exist_ok=True)
receiver_url = input('Local receiver HTTPS tunnel URL: ').strip().rstrip('/')
receiver_token = getpass('Local receiver token: ').strip()
from urllib.parse import urlparse
if urlparse(receiver_url).scheme != 'https':
    raise ValueError('Enter the https:// URL printed by cloudflared')
request = urllib.request.Request(receiver_url+'/bundle', headers={'Authorization':'Bearer '+receiver_token})
with urllib.request.urlopen(request, timeout=120) as response:
    bundle = response.read()
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.infolist():
        target = (ROOT/member.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid bundle path')
    archive.extractall(ROOT)
(ROOT/'runtime').mkdir(exist_ok=True)
connection = ROOT/'runtime/connection.json'
connection.write_text(json.dumps({'url':receiver_url,'token':receiver_token}), encoding='utf-8')
os.chmod(connection, 0o600)
del receiver_token, bundle
sys.path.insert(0, str(ROOT))
print('Bootstrap restored. No API/HF key is stored in notebook output.')

Bootstrap restored. No API/HF key is stored in notebook output.


## 2. Install pinned worker requirements

**Installations and built toolchains are reused within the same Colab runtime. A new runtime reinstalls Python packages but restores the locally saved, pinned CUDA build only when its GPU architecture and CUDA toolkit match.** The frozen base model is downloaded directly from Hugging Face and cached within that runtime. Final quantized artifacts are saved on the laptop.

**Recovery / safe rerun:** After runtime reset rerun 1 then 2; ordinary cell reruns reuse installed packages.

**Cost:** No paid API requests.



In [3]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-colab.txt')],check=True)
print('Worker dependencies ready')

Worker dependencies ready


## 3. Verify receiver, watchdog, configuration and GPU

The same authenticated receiver serves all distinct model notebooks. Check GPU identity and available memory before construction.

**Recovery / safe rerun:** After connection errors rerun 1 and 3. After a new runtime rerun 1–3. Resolve duplicate live worker leases before retrying.

**Cost:** No paid API requests.



In [4]:
from streamlined.common import read_json, verify_experiment, CONDITIONS
from streamlined.transport import Mirror
from continuation.notebook import worker, stream_complete_without_artifact
from continuation.protocol import condition_key
import subprocess
MODEL='qwen7b'
REPRESENTATION='tq1_0'
ADAPTER='joint'
connection=read_json(ROOT/'runtime/connection.json')
mirror=Mirror(connection['url'],connection['token'])
health=mirror.call('/health')
if not health.get('watchdog_alive'):raise RuntimeError('Start the laptop audible watchdog')
print(health, verify_experiment(ROOT)['config_hash'])
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                               '--format=csv,noheader'],text=True))

{'protocol': 'camlds-streamlined-v1', 'watchdog_alive': True} 49e13df7571913d2c39067dfd7dcde93353110b2f4c7523bc8a18de2459b4b73
NVIDIA L4, 23034 MiB, 580.82.07



## 4. Prepare the pinned GGUF artifacts

Merge each complete HF adapter into its frozen BF16 base, convert to GGUF, then construct TQ1_0. The laptop receives each final GGUF artifact and its checksum before the notebook advances. **This stage can spend substantial GPU/CPU time and local disk; completed artifacts and toolchains are reused on rerun within a live runtime.** BitNet SFT export uses a compatibility gate and stops with a clear error if this pinned native path cannot convert the merged checkpoint.

**Recovery / safe rerun:** Rerun 1–3 after a runtime change, then 4. A fully uploaded final artifact is reused; failed conversions leave no completed artifact manifest.

**Cost:** GPU/CPU construction and potentially multi-GB upload; no paid Haiku calls.



On a new L4 runtime, a matching saved `sm_89` Release toolchain and completed GGUF artifacts are restored when available. Quantization is not repeated when the final artifact manifest and checksum are present.

**Parallel use:** assign this `joint` notebook its own Colab runtime. It may share the receiver URL/token with other adapter notebooks because each writes to a distinct adapter stream.

**Lightweight completion guard:** before restoring a multi-GB GGUF, this cell fetches only the small artifact manifest and checks the laptop receiver for terminal rows with the exact frozen configuration identity. A complete stream exits without downloading, hashing or loading the GGUF. Incomplete streams follow the original process unchanged.

In [5]:
from continuation.notebook import stream_complete_without_artifact
complete, coverage = stream_complete_without_artifact(ROOT,MODEL,REPRESENTATION,ADAPTER,mirror)
print('Pre-artifact coverage:',coverage)
if complete:
    print('Adapter stream is complete; skipping GGUF restoration/preparation.')
else:
    worker(ROOT,MODEL,REPRESENTATION,ADAPTER,action='prepare')

Pre-artifact coverage: {'stream': 'quantized/qwen7b/sft_joint/tq1_0', 'expected': 285, 'terminal': 189, 'remaining': 96}
Started quantized/qwen7b/sft_joint/tq1_0/prepare; PID=3073; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen7b_sft_joint_tq1_0_prepare.log
{'model': 'qwen7b', 'representation': 'tq1_0', 'adapter': 'joint', 'artifact_relative': 'models/quantized/qwen7b/sft_joint/tq1_0/model.gguf', 'artifact_sha256': 'e4840eccd580aca2f39f59cd29839fdf611a78063ad61d27edb656605b835810', 'artifact_bytes': 2137344096, 'source_revision': 'a09a35458c702b33eeacc393d103063234e8bc28', 'source_kind': 'frozen_hf', 'adapter_sha256': 'af72c83767941fc11be6ec95fa0873ba46b73b05f291c2243abe1a6983a075b5', 'calibration_sha256': '', 'imatrix_sha256': '', 'calibration_name': '', 'quantization_tensor_counts': {'F32': 141, 'Q4_K': 1, 'Q6_K': 1, 'TQ1_0': 196}, 'toolchain_commit': 'a91eab67d8389d979d29fdf6633b2edd0718eb4b', 'converter_patch': '', 'preparation_seconds': 336.98376325199933}


## 5. Five-row inference pilot

Run five frozen IDs per stream through the final server and save them as real results. The full run skips these IDs. The GGUF tokenizer is compared to the frozen HF tokenizer before generation. Frozen HF token IDs define the experimental input. If the GGUF pre-tokenizer segments identical text differently, the backend must detokenize the frozen IDs back to the exact rendered prompt before generation; the difference is retained in a tokenizer audit. The pilot uses the same per-request context and model-specific parallel-slot policy as the full run. Each row remains a separate request and context. Inspect the server log for actual CUDA offload and token/s before the long run.

**Recovery / safe rerun:** Rerun 1–3 after a new runtime, then 5. Saved pilot IDs are not repeated.

**Cost:** GPU inference for up to five rows per stream.



The pilot retains server verbosity 2 for CUDA-offload and throughput diagnostics. It uses the same slots, batch, microbatch and decoding settings as Section 6.

**Parallel use:** assign this `joint` notebook its own Colab runtime. It may share the receiver URL/token with other adapter notebooks because each writes to a distinct adapter stream.

**Lightweight completion guard:** before restoring a multi-GB GGUF, this cell fetches only the small artifact manifest and checks the laptop receiver for terminal rows with the exact frozen configuration identity. A complete stream exits without downloading, hashing or loading the GGUF. Incomplete streams follow the original process unchanged.

In [ ]:
from continuation.notebook import stream_complete_without_artifact
complete, coverage = stream_complete_without_artifact(ROOT,MODEL,REPRESENTATION,ADAPTER,mirror,limit=5)
print('Pilot coverage:',coverage)
if complete:
    print('Five pilot IDs already terminal; skipping GGUF restoration and pilot server startup.')
else:
    worker(ROOT,MODEL,REPRESENTATION,ADAPTER,action='infer',pilot=5)

## 6. Complete independent inference

Only the `joint` adapter is run on the frozen validation and test IDs. Independent rows are submitted concurrently using a conservative model-specific slot count; completion order may differ, but every result is keyed to its frozen prompt ID. Each response is first written atomically as a JSON record on Colab. A background uploader sends laptop checkpoints in batches of five, applies backpressure at 25 queued rows, and fully flushes at completion. The laptop CSV snapshot is refreshed every 25 saved rows and at normal completion. The same prompt IDs are skipped on resume; errors are retryable. GPU, latency, token and I/O metrics are saved as telemetry.

**Recovery / safe rerun:** After interruption rerun 1–3 and 6. Rerun 4 only if the GGUF artifact was never completed; section 5 is optional on resume. The per-row JSON records, including rows after the latest CSV snapshot, are authoritative.

**Cost:** GPU inference for remaining rows; no paid Haiku calls.



Use one notebook per L4. Missing rows are scheduled from shorter to longer frozen prompt-token buckets through one persistent server. The full run lowers only server log verbosity to 1; response generation and research telemetry are unchanged.

**Parallel use:** assign this `joint` notebook its own Colab runtime. It may share the receiver URL/token with other adapter notebooks because each writes to a distinct adapter stream.

**Lightweight completion guard:** before restoring a multi-GB GGUF, this cell fetches only the small artifact manifest and checks the laptop receiver for terminal rows with the exact frozen configuration identity. A complete stream exits without downloading, hashing or loading the GGUF. Incomplete streams follow the original process unchanged.

In [6]:
from continuation.notebook import stream_complete_without_artifact
complete, coverage = stream_complete_without_artifact(ROOT,MODEL,REPRESENTATION,ADAPTER,mirror)
print('Pre-inference coverage:',coverage)
if complete:
    print('Adapter stream is complete; skipping GGUF restoration, hashing and server startup.')
else:
    worker(ROOT,MODEL,REPRESENTATION,ADAPTER,action='infer')

Pre-inference coverage: {'stream': 'quantized/qwen7b/sft_joint/tq1_0', 'expected': 285, 'terminal': 189, 'remaining': 96}
Started quantized/qwen7b/sft_joint/tq1_0/infer; PID=22469; log=/content/CAM_LDS_STREAMLINED/telemetry/logs/qwen7b_sft_joint_tq1_0_infer.log
Reused laptop CUDA toolchain for architecture 89
quantized/qwen7b/sft_joint/tq1_0: ID 724 saved; 190/285
quantized/qwen7b/sft_joint/tq1_0: ID 137 saved; 192/285
quantized/qwen7b/sft_joint/tq1_0: ID 533 saved; 194/285
quantized/qwen7b/sft_joint/tq1_0: ID 929 saved; 196/285
quantized/qwen7b/sft_joint/tq1_0: ID 284 saved; 198/285
quantized/qwen7b/sft_joint/tq1_0: ID 878 saved; 200/285
quantized/qwen7b/sft_joint/tq1_0: ID 86 saved; 202/285
quantized/qwen7b/sft_joint/tq1_0: ID 599 saved; 204/285
quantized/qwen7b/sft_joint/tq1_0: ID 148 saved; 206/285
quantized/qwen7b/sft_joint/tq1_0: ID 742 saved; 207/285
quantized/qwen7b/sft_joint/tq1_0: ID 940 saved; 208/285
quantized/qwen7b/sft_joint/tq1_0: ID 129 saved; 209/285
quantized/qwen7b/s

: 

## 7. Verify coverage and snapshot

Read current laptop-backed record counts. Check that each required stream is terminal before launching the Haiku judge.

**Recovery / safe rerun:** Rerun 1–3 then 7 after a runtime change. Missing IDs are handled by rerunning 6.

**Cost:** No paid API requests.



**Parallel use:** assign this `joint` notebook its own Colab runtime. It may share the receiver URL/token with other adapter notebooks because each writes to a distinct adapter stream.

In [ ]:
from continuation.store import SyncStore
from continuation.protocol import selected_rows, stream
from collections import Counter
name=stream(MODEL,REPRESENTATION,ADAPTER)
with SyncStore(ROOT,name,mirror) as store:
    store.snapshot()
    expected=len(selected_rows(ROOT,REPRESENTATION,ADAPTER))
    states=Counter(row['status'] for row in store.rows.values())
    print(name,dict(states), 'expected:',expected,'saved:',len(store.rows))